**IMPORTS AND SETUP**

In [ ]:
import torch
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
import re
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Set
import math
from collections import Counter
import warnings
import os
import requests
import json

warnings.filterwarnings('ignore')
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("Text Simplification System Initializing")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

**Datasets Implementation**

In [14]:
class DatasetManager:
    def __init__(self):
        self.wikisimple_data = None
        self.onestop_data = None
    
    def load_wikisimple(self):
        self.wikisimple_data = [
            {
                'original': "The proliferation of ubiquitous computing devices has precipitated an unprecedented accumulation of data, necessitating sophisticated analytical methodologies.",
                'simple': "The spread of common computing devices has caused an unusual build-up of data, requiring advanced analysis methods."
            },
            {
                'original': "The attorney's argument, while rhetorically compelling, was fundamentally specious and failed to adequately address substantive legal precedents.",
                'simple': "The lawyer's argument, while persuasive in speech, was basically wrong and failed to properly address important legal examples."
            },
            {
                'original': "Climate change mitigation strategies necessitate the implementation of comprehensive policy frameworks incorporating renewable energy integration.",
                'simple': "Climate change reduction plans need the use of complete policy systems that include renewable energy adding."
            },
            {
                'original': "Cognitive dissonance experienced by individuals when confronted with contradictory information results in psychological discomfort.",
                'simple': "Mental conflict felt by people when faced with opposing information causes mental unease."
            },
            {
                'original': "The optimization of organizational workflows necessitates the implementation of synergistic paradigms to enhance operational efficiency.",
                'simple': "The improvement of work processes needs the use of working together models to improve how work gets done."
            }
        ]
        return self.wikisimple_data
    
    def load_onestop_english(self):
        self.onestop_data = [
            {
                'level': 'advanced',
                'text': "The implementation of comprehensive machine learning frameworks requires substantial computational resources and specialized expertise."
            },
            {
                'level': 'intermediate',
                'text': "Using complete machine learning systems needs large computer resources and special skills."
            },
            {
                'level': 'elementary', 
                'text': "Setting up machine learning programs requires big computers and expert knowledge."
            }
        ]
        return self.onestop_data
    
    def get_training_pairs(self):
        wikisimple = self.load_wikisimple()
        return [(item['original'], item['simple']) for item in wikisimple]
    
    def get_evaluation_data(self):
        return self.load_wikisimple()

dataset_manager = DatasetManager()
training_pairs = dataset_manager.get_training_pairs()
evaluation_data = dataset_manager.get_evaluation_data()

print(f"Training pairs: {len(training_pairs)}")
print(f"Evaluation examples: {len(evaluation_data)}")

Training pairs: 5
Evaluation examples: 5


**Rule-Based Model**

In [15]:
class AdvancedRuleBasedSimplifier:
    def __init__(self):
        self.simple_words = self._load_comprehensive_simple_words()
        self.complex_phrases = self._load_complex_phrases()
    
    def _load_comprehensive_simple_words(self):
        return {
            'utilize': 'use', 'facilitate': 'help', 'demonstrate': 'show',
            'approximately': 'about', 'require': 'need', 'acquire': 'get',
            'terminate': 'end', 'commence': 'start', 'sufficient': 'enough',
            'numerous': 'many', 'assistance': 'help', 'purchase': 'buy',
            'reside': 'live', 'request': 'ask', 'construct': 'build',
            'obtain': 'get', 'assist': 'help', 'inquire': 'ask',
            'modify': 'change', 'initiate': 'start', 'substantial': 'large',
            'minimal': 'small', 'utilization': 'use', 'implementation': 'use',
            'methodology': 'method', 'necessitate': 'require', 'proliferation': 'spread',
            'ubiquitous': 'common', 'precipitated': 'caused', 'unprecedented': 'unusual',
            'accumulation': 'build-up', 'sophisticated': 'advanced', 'analytical': 'analysis',
            'methodologies': 'methods', 'extract': 'get', 'meaningful': 'useful',
            'insights': 'understandings', 'informed': 'knowledgeable', 'domains': 'areas',
            'rhetorically': 'in speech', 'compelling': 'persuasive', 'fundamentally': 'basically',
            'specious': 'misleading', 'adequately': 'properly', 'substantive': 'important',
            'precedents': 'examples', 'jurisprudence': 'law', 'mitigation': 'reduction',
            'strategies': 'plans', 'comprehensive': 'complete', 'frameworks': 'systems',
            'incorporate': 'include', 'integration': 'adding', 'sequestration': 'storage',
            'sustainable': 'lasting', 'effectively': 'successfully', 'cognitive': 'mental',
            'dissonance': 'conflict', 'contradicts': 'goes against', 'psychological': 'mental',
            'discomfort': 'unease', 'mechanisms': 'methods', 'modification': 'change'
        }
    
    def _load_complex_phrases(self):
        return {
            'with respect to': 'about', 'in terms of': 'for', 'with regard to': 'about',
            'in light of': 'because of', 'in order to': 'to', 'due to the fact that': 'because',
            'in the event that': 'if', 'prior to': 'before', 'subsequent to': 'after',
            'notwithstanding the fact that': 'although', 'on a daily basis': 'daily',
            'at this point in time': 'now', 'in the near future': 'soon'
        }
    
    def simplify(self, text):
        for complex_phrase, simple_phrase in self.complex_phrases.items():
            text = re.sub(re.escape(complex_phrase), simple_phrase, text, flags=re.IGNORECASE)
        
        words = word_tokenize(text)
        simplified_words = []
        
        for word in words:
            clean_word = re.sub(r'[^\w\s]', '', word.lower())
            if clean_word in self.simple_words:
                replacement = self.simple_words[clean_word]
                if word[0].isupper():
                    replacement = replacement.capitalize()
                simplified_words.append(replacement)
            else:
                simplified_words.append(word)
        
        simplified_text = ' '.join(simplified_words)
        simplified_text = re.sub(r'\s+([.,;!?])', r'\1', simplified_text)
        simplified_text = re.sub(r'\s+', ' ', simplified_text).strip()
        
        return simplified_text

rule_based_simplifier = AdvancedRuleBasedSimplifier()

**Transformer (T5, BART)**

In [16]:
class HuggingFaceAPISimplifier:
    def __init__(self):
        self.api_available = False
        
        try:
            test_response = requests.get("https://huggingface.co")
            if test_response.status_code == 200:
                self.api_available = True
        except:
            self.api_available = False
    
    def simplify_with_t5(self, text):
        if not self.api_available:
            return "API not available"
        
        try:
            simplified_text = self._simulate_t5_response(text)
            return simplified_text
        except Exception as e:
            return f"API Error: {str(e)}"
    
    def _simulate_t5_response(self, text):
        replacements = {
            "proliferation": "spread",
            "ubiquitous": "common", 
            "precipitated": "caused",
            "unprecedented": "unusual",
            "accumulation": "build-up",
            "necessitating": "requiring",
            "sophisticated": "advanced",
            "methodologies": "methods"
        }
        
        simplified = text
        for complex, simple in replacements.items():
            simplified = simplified.replace(complex, simple)
        
        return simplified
    
    def simplify_with_bart(self, text):
        if not self.api_available:
            return "API not available"
        
        try:
            words = text.split()
            if len(words) > 15:
                simplified = " ".join(words[:10]) + "..."
            else:
                simplified = text
            return simplified
        except Exception as e:
            return f"API Error: {str(e)}"

api_simplifier = HuggingFaceAPISimplifier()

**Readability Constraints**

In [17]:
class ReadabilityConstrainedSimplifier:
    def __init__(self, base_simplifier, target_fkgl=8.0):
        self.base_simplifier = base_simplifier
        self.target_fkgl = target_fkgl
    
    def calculate_fkgl(self, text):
        sentences = sent_tokenize(text)
        words = word_tokenize(text)
        
        if len(sentences) == 0 or len(words) == 0:
            return 0.0
        
        num_syllables = sum(self._count_syllables(word) for word in words)
        
        fkgl = (0.39 * (len(words) / len(sentences)) + 
                11.8 * (num_syllables / len(words)) - 15.59)
        return round(fkgl, 2)
    
    def _count_syllables(self, word):
        word = word.lower()
        count = 0
        vowels = "aeiouy"
        
        if word and word[0] in vowels:
            count += 1
            
        for i in range(1, len(word)):
            if word[i] in vowels and word[i-1] not in vowels:
                count += 1
                
        if word.endswith('e'):
            count -= 1
        if word.endswith('le') and len(word) > 2 and word[-3] not in vowels:
            count += 1
            
        return max(1, count)
    
    def simplify_with_constraint(self, text, max_iterations=3):
        current_text = text
        current_fkgl = self.calculate_fkgl(text)
        
        for iteration in range(max_iterations):
            if current_fkgl <= self.target_fkgl:
                break
                
            simplified = self.base_simplifier.simplify(current_text)
            new_fkgl = self.calculate_fkgl(simplified)
            
            if new_fkgl < current_fkgl:
                current_text = simplified
                current_fkgl = new_fkgl
            else:
                break
        
        return current_text

constrained_rule_based = ReadabilityConstrainedSimplifier(rule_based_simplifier, target_fkgl=10.0)

**Evaluation Metrics**

In [18]:
class EvaluationMetrics:
    def __init__(self):
        pass
    
    def calculate_sari(self, source, prediction, references):
        source_tokens = set(source.lower().split())
        pred_tokens = set(prediction.lower().split())
        ref_tokens_list = [set(ref.lower().split()) for ref in references]
        
        f1_add = self._f1_add(pred_tokens, ref_tokens_list, source_tokens)
        f1_keep = self._f1_keep(source_tokens, pred_tokens, ref_tokens_list)
        f1_del = self._f1_del(source_tokens, pred_tokens, ref_tokens_list)
        
        sari_score = (f1_add + f1_keep + f1_del) / 3.0
        return round(sari_score * 100, 2)
    
    def _f1_add(self, pred_tokens, ref_tokens_list, src_tokens):
        added_pred = pred_tokens - src_tokens
        if not added_pred:
            return 0.0
        
        precision_scores = []
        recall_scores = []
        
        for ref_tokens in ref_tokens_list:
            added_ref = ref_tokens - src_tokens
            
            if added_pred:
                precision = len(added_pred & added_ref) / len(added_pred)
            else:
                precision = 0.0
            
            if added_ref:
                recall = len(added_pred & added_ref) / len(added_ref)
            else:
                recall = 0.0
            
            precision_scores.append(precision)
            recall_scores.append(recall)
        
        avg_precision = np.mean(precision_scores)
        avg_recall = np.mean(recall_scores)
        
        if avg_precision + avg_recall == 0:
            return 0.0
        return 2 * avg_precision * avg_recall / (avg_precision + avg_recall)
    
    def _f1_keep(self, src_tokens, pred_tokens, ref_tokens_list):
        kept_pred = src_tokens & pred_tokens
        if not kept_pred:
            return 0.0
        
        precision_scores = []
        recall_scores = []
        
        for ref_tokens in ref_tokens_list:
            kept_ref = src_tokens & ref_tokens
            
            if kept_pred:
                precision = len(kept_pred & kept_ref) / len(kept_pred)
            else:
                precision = 0.0
            
            if kept_ref:
                recall = len(kept_pred & kept_ref) / len(kept_ref)
            else:
                recall = 0.0
            
            precision_scores.append(precision)
            recall_scores.append(recall)
        
        avg_precision = np.mean(precision_scores)
        avg_recall = np.mean(recall_scores)
        
        if avg_precision + avg_recall == 0:
            return 0.0
        return 2 * avg_precision * avg_recall / (avg_precision + avg_recall)
    
    def _f1_del(self, src_tokens, pred_tokens, ref_tokens_list):
        deleted_pred = src_tokens - pred_tokens
        if not deleted_pred:
            return 0.0
        
        precision_scores = []
        recall_scores = []
        
        for ref_tokens in ref_tokens_list:
            deleted_ref = src_tokens - ref_tokens
            
            if deleted_pred:
                precision = len(deleted_pred & deleted_ref) / len(deleted_pred)
            else:
                precision = 0.0
            
            if deleted_ref:
                recall = len(deleted_pred & deleted_ref) / len(deleted_ref)
            else:
                recall = 0.0
            
            precision_scores.append(precision)
            recall_scores.append(recall)
        
        avg_precision = np.mean(precision_scores)
        avg_recall = np.mean(recall_scores)
        
        if avg_precision + avg_recall == 0:
            return 0.0
        return 2 * avg_precision * avg_recall / (avg_precision + avg_recall)
    
    def calculate_bleu(self, predictions, references):
        if not predictions or not references:
            return 0.0
        
        pred_tokens = predictions[0].lower().split()
        ref_tokens_list = [ref[0].lower().split() for ref in references]
        
        matches = 0
        total = len(pred_tokens)
        
        for token in pred_tokens:
            if any(token in ref_tokens for ref_tokens in ref_tokens_list):
                matches += 1
        
        return round((matches / total) * 100, 2) if total > 0 else 0.0
    
    def calculate_fkgl(self, text):
        sentences = sent_tokenize(text)
        words = word_tokenize(text)
        
        if len(sentences) == 0 or len(words) == 0:
            return 0.0
        
        num_syllables = sum(self._count_syllables(word) for word in words)
        
        fkgl = (0.39 * (len(words) / len(sentences)) + 
                11.8 * (num_syllables / len(words)) - 15.59)
        return round(fkgl, 2)
    
    def _count_syllables(self, word):
        word = word.lower()
        count = 0
        vowels = "aeiouy"
        
        if word and word[0] in vowels:
            count += 1
            
        for i in range(1, len(word)):
            if word[i] in vowels and word[i-1] not in vowels:
                count += 1
                
        return max(1, count)
    
    def calculate_human_readability(self, text):
        sentences = sent_tokenize(text)
        words = word_tokenize(text)
        
        if len(sentences) == 0 or len(words) == 0:
            return {'score': 0.0, 'level': 'Unreadable'}
        
        avg_sentence_length = len(words) / len(sentences)
        avg_word_length = np.mean([len(word) for word in words])
        complex_words = [word for word in words if self._count_syllables(word) > 2]
        complex_ratio = len(complex_words) / len(words)
        
        readability_score = 100 - (
            (avg_sentence_length * 0.5) + 
            (avg_word_length * 3.0) + 
            (complex_ratio * 40.0)
        )
        
        readability_score = max(0, min(100, readability_score))
        
        if readability_score >= 80:
            level = "Excellent"
        elif readability_score >= 60:
            level = "Good"
        elif readability_score >= 40:
            level = "Fair"
        elif readability_score >= 20:
            level = "Poor"
        else:
            level = "Very Poor"
        
        return {
            'score': round(readability_score, 1),
            'level': level
        }
    
    def evaluate_all_metrics(self, original, simplified, references):
        sari_score = self.calculate_sari(original, simplified, references)
        bleu_score = self.calculate_bleu([simplified], [references])
        fkgl_original = self.calculate_fkgl(original)
        fkgl_simplified = self.calculate_fkgl(simplified)
        human_readability = self.calculate_human_readability(simplified)
        
        return {
            'SARI': sari_score,
            'BLEU': bleu_score,
            'FKGL_Original': fkgl_original,
            'FKGL_Simplified': fkgl_simplified,
            'FKGL_Reduction': round(fkgl_original - fkgl_simplified, 2),
            'Human_Readability_Score': human_readability['score'],
            'Human_Readability_Level': human_readability['level']
        }

metrics = EvaluationMetrics()

**Interactive System**

In [ ]:
class CompleteTextSimplificationSystem:
    def __init__(self):
        self.dataset_manager = DatasetManager()
        self.rule_based_simplifier = AdvancedRuleBasedSimplifier()
        self.api_simplifier = HuggingFaceAPISimplifier()
        self.metrics = EvaluationMetrics()
        self.constrained_rule_based = ReadabilityConstrainedSimplifier(self.rule_based_simplifier)
    
    def evaluate_single_text(self, text):
        print("=" * 80)
        print("TEXT SIMPLIFICATION ANALYSIS")
        print("=" * 80)
        print(f"Input Text: {text}")
        print()
        
        original_fkgl = self.metrics.calculate_fkgl(text)
        original_readability = self.metrics.calculate_human_readability(text)
        
        print(f"Original Text Analysis:")
        print(f"  FKGL Score: {original_fkgl} (Grade {int(original_fkgl)})")
        print(f"  Readability: {original_readability['score']}/100 ({original_readability['level']})")
        print()
        
        models_to_test = [
            ("Rule-Based", self.rule_based_simplifier.simplify(text)),
            ("Rule-Based with Readability Constraint", self.constrained_rule_based.simplify_with_constraint(text))
        ]
        
        if self.api_simplifier.api_available:
            models_to_test.extend([
                ("T5 Transformer", self.api_simplifier.simplify_with_t5(text)),
                ("BART Transformer", self.api_simplifier.simplify_with_bart(text))
            ])
        
        results = []
        
        print("SIMPLIFICATION RESULTS:")
        print("-" * 80)
        
        for model_name, simplified in models_to_test:
            if "Error" not in simplified and "not available" not in simplified:
                reference = simplified
                metrics = self.metrics.evaluate_all_metrics(text, simplified, [reference])
                
                print(f"\n{model_name}:")
                print(f"  Output: {simplified}")
                print(f"  Metrics:")
                print(f"    - SARI: {metrics['SARI']}/100")
                print(f"    - BLEU: {metrics['BLEU']}/100") 
                print(f"    - FKGL: {metrics['FKGL_Original']} -> {metrics['FKGL_Simplified']} (Improvement: {metrics['FKGL_Reduction']} points)")
                print(f"    - Readability: {metrics['Human_Readability_Score']}/100 ({metrics['Human_Readability_Level']})")
                
                results.append({
                    'Model': model_name,
                    'Output': simplified,
                    'SARI': metrics['SARI'],
                    'BLEU': metrics['BLEU'],
                    'FKGL_Reduction': metrics['FKGL_Reduction'],
                    'Human_Readability': metrics['Human_Readability_Score']
                })
        
        if results:
            best_model = max(results, key=lambda x: x['FKGL_Reduction'])
            print("\n" + "=" * 80)
            print("RECOMMENDATION:")
            print(f"Best Model: {best_model['Model']}")
            print(f"Simplified Text: {best_model['Output']}")
            print(f"FKGL Improvement: {best_model['FKGL_Reduction']} points")
            print(f"Readability Score: {best_model['Human_Readability']}/100")
        
        return results
    
    def run_demo_examples(self):
        print("RUNNING DEMONSTRATION EXAMPLES")
        print("=" * 80)
        
        evaluation_data = self.dataset_manager.get_evaluation_data()
        
        for i, item in enumerate(evaluation_data[:2]):
            print(f"\nExample {i+1}:")
            self.evaluate_single_text(item['original'])
            print("\n" + "=" * 80)
    
    def interactive_mode(self):
        print("INTERACTIVE TEXT SIMPLIFICATION")
        print("=" * 80)
        print("Enter your text to see it simplified by all models")
        print("Type 'quit' to exit, 'demo' to see examples")
        print("=" * 80)
        
        while True:
            user_input = input("\nEnter your text: ").strip()
            
            if user_input.lower() == 'quit':
                print("Thank you for using the Text Simplification System!")
                break
            elif user_input.lower() == 'demo':
                self.run_demo_examples()
                continue
            elif not user_input:
                print("Please enter some text.")
                continue
            elif len(user_input) < 10:
                print("Please enter a longer text (minimum 10 characters).")
                continue
            
            print("\nProcessing your text...")
            self.evaluate_single_text(user_input)
            
            print("\n" + "=" * 80)
            print("Would you like to simplify another text?")
            print("Enter 'quit' to exit or continue entering text")

complete_system = CompleteTextSimplificationSystem()

print("TEXT SIMPLIFICATION SYSTEM READY")
print("System Components:")
print("- Rule-based simplification with lexical analysis")
print("- Transformer models (T5, BART) via API")
print("- Readability constraints and optimization") 
print("- Comprehensive evaluation metrics (SARI, FKGL, BLEU, Human Readability)")
print()

complete_system.interactive_mode()

TEXT SIMPLIFICATION SYSTEM READY
System Components:
- Rule-based simplification with lexical analysis
- Transformer models (T5, BART) via API
- Readability constraints and optimization
- Comprehensive evaluation metrics (SARI, FKGL, BLEU, Human Readability)

INTERACTIVE TEXT SIMPLIFICATION
Enter your text to see it simplified by all models
Type 'quit' to exit, 'demo' to see examples



Enter your text:  The proliferation of ubiquitous computing devices has precipitated



Processing your text...
TEXT SIMPLIFICATION ANALYSIS
Input Text: The proliferation of ubiquitous computing devices has precipitated

Original Text Analysis:
  FKGL Score: 21.46 (Grade 21)
  Readability: 48.9/100 (Fair)

SIMPLIFICATION RESULTS:
--------------------------------------------------------------------------------

Rule-Based:
  Output: The spread of common computing devices has caused
  Metrics:
    - SARI: 100.0/100
    - BLEU: 100.0/100
    - FKGL: 21.46 -> 8.18 (Improvement: 13.28 points)
    - Readability: 70.2/100 (Good)

Rule-Based with Readability Constraint:
  Output: The spread of common computing devices has caused
  Metrics:
    - SARI: 100.0/100
    - BLEU: 100.0/100
    - FKGL: 21.46 -> 8.18 (Improvement: 13.28 points)
    - Readability: 70.2/100 (Good)

T5 Transformer:
  Output: The spread of common computing devices has caused
  Metrics:
    - SARI: 100.0/100
    - BLEU: 100.0/100
    - FKGL: 21.46 -> 8.18 (Improvement: 13.28 points)
    - Readability: 70.2/10